# 07 — Interactive Member Selection

Reproduces the validator photometry panels (`plots_validate_cmds` / `plots_validate_cc`)
plus VPD and parallax planes as **interactive plotly panels**, and lets you draw
box/lasso selections to mark trustworthy members.

**How to use**
1. Run all cells. Each panel supports box select (default) and lasso (toolbar).
2. Selections on different panels **intersect**: a star must fall inside the
   selection of *every* panel you have drawn on. Panels without a selection
   don't constrain. Draw again on a panel to replace its selection
   (shift-drag to *extend* a panel's selection).
3. Selected stars are highlighted in every panel; the live counter shows the
   combined count.
4. Press **Save member_seed.csv**, then run the **final summary cell** to get
   static plots of the selected members across every panel (also saved as
   `member_seed_selection.png`), and run the pop fit with
   `--member_seed_csv <field>/member_seed.csv`.

Requires `plotly` + `anywidget` (`pip install plotly anywidget`).

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = '..'
FIELD_NAME = 'Sculptor_dSph'
SEED_OUT   = 'member_seed.csv'   # written into the field directory
VPD_ZOOM   = 3.0                 # initial VPD half-width (mas/yr)
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
from pathlib import Path

field_dir = Path(OUTPUT_DIR).expanduser().resolve() / FIELD_NAME
print('field:', field_dir)

In [ ]:
# ── Load: v1 astrometry + validator photometry catalogue ────────────────────
# Gaia IDs are 19-digit int64: always read them with an explicit dtype and
# carry them as STRINGS through plotly (JS numbers corrupt ints > 2^53).
sa = pd.read_csv(field_dir / 'BP3M_results' / 'stellar_astrometry.csv',
                 dtype={'Gaia_id': np.int64})

wide = None
cmc_path = field_dir / 'cross_match_catalog.csv'
if cmc_path.exists():
    cat = pd.read_csv(cmc_path, dtype={'gaia_source_id': np.int64})
    wide = (cat.pivot_table(index='gaia_source_id', columns='filter_camera',
                            values='mag_norm_wmean', aggfunc='first')
               .reset_index())
    filter_cols = [c for c in wide.columns if c != 'gaia_source_id']
    wide = wide.rename(columns={c: f'mag_wmean_{c}' for c in filter_cols})
    print('HST filters:', filter_cols)
else:
    print('cross_match_catalog.csv not found — photometry panels will be Gaia-only')

master = sa.rename(columns={'Gaia_id': 'gaia_source_id'})
if wide is not None:
    master = master.merge(wide, on='gaia_source_id', how='left')
master['gid_str'] = master['gaia_source_id'].astype(str)
print(f'{len(master)} stars loaded')

In [ ]:
# ── Panel definitions ────────────────────────────────────────────────────────
# Mirrors _plot_cmds / _plot_color_color: HST bands sorted blue→red, each pair
# gives colour (blue−red) vs the redder magnitude; ≥3 bands add colour-colour.
_WAVELENGTHS = {  # nm, for blue→red ordering (same table as _plot_cmds)
    'F275W': 275, 'F336W': 336, 'F390W': 390, 'F435W': 435, 'F438W': 438,
    'F475W': 475, 'F555W': 555, 'F606W': 606, 'F625W': 625, 'F775W': 775,
    'F814W': 814, 'F850LP': 900, 'F110W': 1100, 'F125W': 1250, 'F160W': 1600,
}

def _wl(band):
    key = band.split('/')[0].upper()
    return _WAVELENGTHS.get(key, 9999)

mag_cols = sorted((c for c in master.columns if c.startswith('mag_wmean_')),
                  key=lambda c: _wl(c.replace('mag_wmean_', '')))
bands = [c.replace('mag_wmean_', '') for c in mag_cols]

panels = []   # (title, x, y, xlabel, ylabel, invert_y)

# VPD — the primary member-selection plane
panels.append(('VPD (v1 bp3m)', master['pmra_bp3m'], master['pmdec_bp3m'],
               'pmra [mas/yr]', 'pmdec [mas/yr]', False))

# Gaia CMD
if {'gmag', 'bp_rp'}.issubset(master.columns):
    panels.append(('Gaia CMD', master['bp_rp'], master['gmag'],
                   'BP − RP', 'G', True))

# HST CMDs: every blue/red pair
for i in range(len(bands)):
    for j in range(i + 1, len(bands)):
        b, r = bands[i], bands[j]
        panels.append((f'{b} − {r} CMD',
                       master[f'mag_wmean_{b}'] - master[f'mag_wmean_{r}'],
                       master[f'mag_wmean_{r}'],
                       f'{b} − {r}', r, True))

# Colour-colour (needs ≥3 HST bands)
if len(bands) >= 3:
    for i in range(len(bands) - 2):
        b1, b2, b3 = bands[i], bands[i + 1], bands[i + 2]
        panels.append((f'({b1}−{b2}) vs ({b2}−{b3})',
                       master[f'mag_wmean_{b1}'] - master[f'mag_wmean_{b2}'],
                       master[f'mag_wmean_{b2}'] - master[f'mag_wmean_{b3}'],
                       f'{b1} − {b2}', f'{b2} − {b3}', False))

# Parallax plane
if 'parallax_bp3m' in master.columns:
    panels.append(('Parallax', master['gmag'], master['parallax_bp3m'],
                   'G', 'parallax [mas]', False))

print(f'{len(panels)} panels: ' + ', '.join(p[0] for p in panels))

In [ ]:
# ── Interactive selection UI ─────────────────────────────────────────────────
import plotly.graph_objects as go
import ipywidgets as W

gid_all   = master['gid_str'].to_numpy()
panel_sel = {}          # panel index -> set of gid_str (absent/None = no constraint)
figs      = []

status = W.HTML()

def combined_selection():
    active = [s for s in panel_sel.values() if s is not None]
    if not active:
        return None                       # no constraint anywhere
    out = set(gid_all)
    for s in active:
        out &= s
    return out

def refresh():
    comb = combined_selection()
    for fw in figs:
        tr = fw.data[0]
        if comb is None:
            tr.selectedpoints = None
        else:
            ids = tr.customdata[:, 0]
            tr.selectedpoints = [k for k, g in enumerate(ids) if g in comb]
    n = len(comb) if comb is not None else len(gid_all)
    n_active = sum(1 for s in panel_sel.values() if s is not None)
    status.value = (f'<b>{n}</b> stars selected '
                    f'({n_active} panel constraint{"s" if n_active != 1 else ""} active)')

def _make_handler(panel_idx):
    def _on_select(trace, points, selector):
        ids = {trace.customdata[k][0] for k in points.point_inds}
        prev = panel_sel.get(panel_idx)
        # shift-drag extends this panel's selection; plain drag replaces it
        if getattr(selector, 'shift', False) and prev is not None:
            ids |= prev
        panel_sel[panel_idx] = ids if ids else None
        refresh()
    return _on_select

for k, (title, x, y, xl, yl, inv) in enumerate(panels):
    fin = np.isfinite(np.asarray(x, float)) & np.isfinite(np.asarray(y, float))
    fw = go.FigureWidget(
        data=[go.Scattergl(
            x=np.asarray(x, float)[fin], y=np.asarray(y, float)[fin],
            mode='markers',
            marker=dict(size=4, color='#1f77b4'),
            customdata=np.c_[gid_all[fin]],
            selected=dict(marker=dict(color='#d62728', size=6)),
            unselected=dict(marker=dict(opacity=0.25)),
            hovertemplate='%{customdata[0]}<extra></extra>',
        )],
        layout=go.Layout(
            title=dict(text=title, font=dict(size=13)),
            width=430, height=380, dragmode='select',
            margin=dict(l=55, r=10, t=40, b=45),
            xaxis=dict(title=xl), yaxis=dict(title=yl),
        ),
    )
    if inv:
        fw.layout.yaxis.autorange = 'reversed'
    if title.startswith('VPD'):
        fw.layout.xaxis.range = [-VPD_ZOOM, VPD_ZOOM]
        fw.layout.yaxis.range = [-VPD_ZOOM, VPD_ZOOM]
    fw.data[0].on_selection(_make_handler(k))
    figs.append(fw)

def _save(_btn=None):
    comb = combined_selection()
    if comb is None:
        status.value = '<b style="color:red">Nothing selected — draw a box first.</b>'
        return
    out = pd.DataFrame({
        'gaia_source_id': sorted(np.int64(g) for g in comb),
        'trusted': True,
    })
    out_path = field_dir / SEED_OUT
    out.to_csv(out_path, index=False)
    status.value = f'<b style="color:green">Saved {len(out)} members → {out_path}</b>'

def _clear(_btn=None):
    panel_sel.clear()
    refresh()

save_btn  = W.Button(description='Save member_seed.csv', button_style='success')
clear_btn = W.Button(description='Clear all selections')
save_btn.on_click(_save)
clear_btn.on_click(_clear)

refresh()
rows = [W.HBox(figs[i:i + 2]) for i in range(0, len(figs), 2)]
W.VBox([W.HBox([save_btn, clear_btn, status]), *rows])

In [ ]:
# ── Final selection summary (run AFTER drawing/saving your selection) ───────
# Static matplotlib version of every panel with the selected members
# highlighted. Uses the live selection if one exists, otherwise falls back to
# the saved member_seed.csv, so this cell also works in a fresh kernel.
import matplotlib.pyplot as plt

comb = combined_selection()
if comb is None and (field_dir / SEED_OUT).exists():
    _saved = pd.read_csv(field_dir / SEED_OUT, dtype={'gaia_source_id': np.int64})
    if 'trusted' in _saved.columns:
        _saved = _saved[_saved['trusted'].astype(bool)]
    comb = set(_saved['gaia_source_id'].astype(str))
    print(f'No live selection — loaded {len(comb)} members from {SEED_OUT}')

if comb is None:
    print('No selection drawn and no saved seed CSV — nothing to summarise.')
else:
    sel_mask = master['gid_str'].isin(comb).to_numpy()
    ncol = 3
    nrow = int(np.ceil(len(panels) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 4.0 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, (title, x, y, xl, yl, inv) in zip(axes, panels):
        xv = np.asarray(x, float); yv = np.asarray(y, float)
        fin = np.isfinite(xv) & np.isfinite(yv)
        ax.scatter(xv[fin & ~sel_mask], yv[fin & ~sel_mask],
                   s=4, c='0.75', lw=0, label='not selected')
        ax.scatter(xv[fin & sel_mask], yv[fin & sel_mask],
                   s=8, c='crimson', lw=0, label='selected member')
        ax.set_xlabel(xl); ax.set_ylabel(yl)
        ax.set_title(f'{title}  ({int((fin & sel_mask).sum())} sel)', fontsize=10)
        if inv:
            ax.invert_yaxis()
        if title.startswith('VPD'):
            ax.set_xlim(-VPD_ZOOM, VPD_ZOOM); ax.set_ylim(-VPD_ZOOM, VPD_ZOOM)
    for ax in axes[len(panels):]:
        ax.set_visible(False)
    axes[0].legend(fontsize=8, loc='upper right')
    fig.suptitle(f'{FIELD_NAME} — member selection: '
                 f'{int(sel_mask.sum())} of {len(master)} stars', y=1.001)
    fig.tight_layout()
    out_png = field_dir / 'member_seed_selection.png'
    fig.savefig(out_png, dpi=150, bbox_inches='tight')
    print(f'Saved summary figure → {out_png}')
    plt.show()

## Using the selection

```bash
bp3m-pop-fit --name FIELD --lvd_key KEY --member_seed_csv FIELD/member_seed.csv [...]
```

The seed **replaces** the automatic sigma-clip initial member selection: it
defines the starting member set and the μ_pop prior centre. The pop-fit
phases still refine membership statistically from there — the seed is a
starting point, not a hard lock.